# ScentAI Stage 6 - Drive to Modal artifact upload

This notebook uploads the full LoRA adapter, Chroma snapshot, and SQLite catalog directly from Google Drive to Modal Volumes. It does not start a GPU.

You need a Modal API token pair (`MODAL_TOKEN_ID` and `MODAL_TOKEN_SECRET`). The secret is the private, password-like half of the pair and is shown only when the token is created. It is not the ScentAI API key or Hugging Face token.

In [ ]:
%pip install -q modal==1.5.2

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import getpass
import os
import subprocess

os.environ['MODAL_TOKEN_ID'] = getpass.getpass('Modal token ID: ').strip()
os.environ['MODAL_TOKEN_SECRET'] = getpass.getpass('Modal token secret: ').strip()
assert os.environ['MODAL_TOKEN_ID'], 'Modal token ID is required.'
assert os.environ['MODAL_TOKEN_SECRET'], 'Modal token secret is required.'
subprocess.run(['modal', 'token', 'info'], check=True)
print('Modal authentication: OK')

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Perfume-Dataset')
ADAPTER_DIR = PROJECT_ROOT / 'models/scentai-gemma-4-12b-it-full-fastmodel-lora/best_lora_adapter'
CHROMA_DIR = PROJECT_ROOT / 'chroma_db_bge_m3'
CATALOG_PATH = PROJECT_ROOT / 'scentai_catalog.sqlite3'

required_paths = {
    'adapter config': ADAPTER_DIR / 'adapter_config.json',
    'adapter weights': ADAPTER_DIR / 'adapter_model.safetensors',
    'Chroma database': CHROMA_DIR / 'chroma.sqlite3',
    'catalog': CATALOG_PATH,
}
missing = [f'{name}: {path}' for name, path in required_paths.items() if not path.is_file()]
assert not missing, 'Missing Drive artifacts:\n' + '\n'.join(missing)
for name, path in required_paths.items():
    print(f'{name:18s} | {path} | {path.stat().st_size / (1024 ** 2):.1f} MiB')
print('Drive artifact validation: OK')

In [ ]:
def ensure_volume(name):
    result = subprocess.run(
        ['modal', 'volume', 'create', name],
        text=True,
        capture_output=True,
    )
    message = (result.stdout or '') + (result.stderr or '')
    if result.returncode and 'already exists' not in message.lower():
        raise RuntimeError(f'Could not create {name}: {message}')
    print('Volume ready:', name)

ensure_volume('scentai-models')
ensure_volume('scentai-data')

uploads = [
    ('scentai-models', ADAPTER_DIR, '/scentai'),
    ('scentai-data', CHROMA_DIR, '/chroma_db_bge_m3'),
    ('scentai-data', CATALOG_PATH, '/scentai_catalog.sqlite3'),
]
for volume, source, destination in uploads:
    print(f'Uploading {source.name} -> {volume}:{destination}', flush=True)
    subprocess.run(
        ['modal', 'volume', 'put', '--force', volume, str(source), destination],
        check=True,
    )

subprocess.run(['modal', 'volume', 'ls', 'scentai-models', '/scentai'], check=True)
subprocess.run(['modal', 'volume', 'ls', 'scentai-data', '/'], check=True)
print('\nArtifact upload complete. Run the Modal artifact preflight next.')